# Làm sạch và chuẩn bị dữ liệu


**Dataset:** Crop Recommendation Dataset

**Vai trò:** Thành viên 2 - Data Engineer

### Mục tiêu

- Kiểm tra và xử lý dữ liệu thiếu.
- Kiểm tra và xử lý các giá trị ngoại lệ (Outlier).
- Mã hóa biến mục tiêu `label` từ dạng chữ sang dạng số.
- Tách dữ liệu thành tập huấn luyện (Train) và tập kiểm thử (Test).
- Chuẩn hóa các đặc trưng đầu vào.
- Xuất dữ liệu đã được tiền xử lý để các thành viên tiếp tục xây dựng và đánh giá mô hình Machine Learning.

Kết quả phân tích từ 01_eda_and_visualization cho thấy:

- Dataset có 2.200 dòng và 8 cột.
- Không có giá trị thiếu.
- Không có dữ liệu trùng lặp.
- Có 22 loại cây và mỗi loại có 100 mẫu.
- Có Outlier ở các biến P, K, temperature, humidity, ph và rainfall.
- Biến N không phát hiện Outlier theo phương pháp IQR.
- Các biến P, K, temperature, humidity, ph và rainfall có xuất hiện Outlier theo phương pháp IQR.

Tuy nhiên, các giá trị được xác định là Outlier theo IQR không nhất thiết là dữ liệu sai. Trong bài toán dự đoán cây trồng, các giá trị này có thể phản ánh những điều kiện đất và môi trường đặc trưng cho từng loại cây.

Do đó, không loại bỏ trực tiếp các dòng chứa Outlier. Thay vào đó, sử dụng phương pháp IQR Capping để giới hạn ảnh hưởng của các giá trị quá cực đoan nhưng vẫn giữ lại các mẫu dữ liệu.

Sau khi xử lý Outlier, dữ liệu sẽ được mã hóa biến mục tiêu, chia thành tập Train/Test và chuẩn hóa các đặc trưng trước khi đưa vào mô hình Machine Learning.

In [1]:
import pandas as pd  # thư viện đọc, xử lý và phân tích dl dạng bảng
import numpy as np  # thư viện làm việc với mảng nhiều chiều

from sklearn.preprocessing import LabelEncoder, StandardScaler # 2 công cụ Mã hóa nhãn và chuẩn hóa dl
from sklearn.model_selection import train_test_split

print("Các thư viện đã được import thành công!")

Các thư viện đã được import thành công!


## 1. Đọc dữ liệu

Dữ liệu được sử dụng là bộ dữ liệu đã được phân tích ở bước 01.

Theo kết quả phân tích, các đặc trưng được lựa chọn gồm:

- `N`: Nitơ.
- `P`: Photpho.
- `K`: Kali.
- `temperature`: Nhiệt độ.
- `humidity`: Độ ẩm.
- `ph`: Độ pH.
- `rainfall`: Lượng mưa.

Biến `label` là biến mục tiêu, đại diện cho loại cây trồng.

In [2]:

data_path = "../data/raw/Crop_recommendation.csv"

df = pd.read_csv(data_path)

selected_columns = [
    "N",
    "P",
    "K",
    "temperature",
    "humidity",
    "ph",
    "rainfall",
    "label"
]

df = df[selected_columns].copy() # tạo ra thêm 1 bản sao chứa các đặt trưng

print("Đọc dữ liệu thành công!")
print("Kích thước dữ liệu:", df.shape)

display(df.head()) # hiển thị 5 dòng đầu tiên của df

Đọc dữ liệu thành công!
Kích thước dữ liệu: (2200, 8)


,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


### Nhận xét

Dữ liệu được đọc thành công và có kích thước 2.200 dòng và 8 cột.

Dataset gồm 7 đặc trưng đầu vào về đất và điều kiện môi trường, cùng với biến mục tiêu `label` biểu diễn loại cây trồng.

Dữ liệu được giữ nguyên để tiếp tục thực hiện các bước kiểm tra chất lượng và tiền xử lý.

### Nhận xét:

Dataset có 2.200 dòng và 8 cột,các đặc trưng đầu vào đều ở dạng dữ liệu số, phù hợp để thực hiện các bước xử lý Outlier và chuẩn hóa.

Biến `label` ở dạng chuỗi, do đó cần được mã hóa sang dạng số trước khi đưa vào mô hình Machine Learning.

# 2. Xác định biến đầu vào và biến mục tiêu

Trong bài toán Machine Learning, dữ liệu được chia thành:

- X: các biến đầu vào được sử dụng để dự đoán.
- y: biến mục tiêu mà mô hình cần dự đoán.

Các biến đầu vào gồm:

N, P, K, temperature, humidity, ph và rainfall.

Biến mục tiêu là `label`.

In [3]:
features = [
    "N",
    "P",
    "K",
    "temperature",
    "humidity",
    "ph",
    "rainfall"
]

target = "label"

X = df[features].copy()
y = df[target].copy()

print("Các đặc trưng đầu vào:")
print(features)

print("\nBiến mục tiêu:")
print(target)

print("\nKích thước X:", X.shape)
print("Kích thước y:", y.shape)

Các đặc trưng đầu vào:
['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']

Biến mục tiêu:
label

Kích thước X: (2200, 7)
Kích thước y: (2200,)


## 3. Tách dữ liệu đầu vào X và biến mục tiêu y

Dữ liệu được chia thành:

- X: chứa 7 đặc trưng đầu vào.
- y: chứa biến mục tiêu `label`.

Đây là bước chuẩn bị trước khi chia tập Train/Test.

In [4]:
X = df[features].copy()
y = df[target].copy()

print("Kích thước X:", X.shape)
print("Kích thước y:", y.shape)

display(X.head())
display(y.head())

Kích thước X: (2200, 7)
Kích thước y: (2200,)


,N,P,K,temperature,humidity,ph,rainfall
0,90,42,43,20.879744,82.002744,6.502985,202.935536
1,85,58,41,21.770462,80.319644,7.038096,226.655537
2,60,55,44,23.004459,82.320763,7.840207,263.964248
3,74,35,40,26.491096,80.158363,6.980401,242.864034
4,78,42,42,20.130175,81.604873,7.628473,262.717340


0    rice
1    rice
2    rice
3    rice
4    rice
Name: label, dtype: str

## 4. Mã hóa biến mục tiêu bằng Label Encoding

Biến `label` hiện đang ở dạng chuỗi, ví dụ:

- apple
- banana
- maize
- rice
- watermelon

Các thuật toán Machine Learning cần biến mục tiêu ở dạng số.

Do đó sử dụng `LabelEncoder` để chuyển các nhãn chữ thành các giá trị số từ 0 đến 21.

Việc mã hóa này chỉ áp dụng cho biến mục tiêu `y`, không áp dụng cho các đặc trưng số.

Hàm fit_transform(y):
- fit (học): Nó sẽ đọc qua toàn bộ các chữ có trong y, tìm ra danh sách tất cả các loại cây khác nhau (ví dụ có 22 loại) và tự động gán cho mỗi loại một con số từ 0 đến 21 theo thứ tự bảng chữ cái.
- transform (biến đổi): Nó thay thế toàn bộ các chữ trong tập y bằng các con số tương ứng vừa được gán.

In [5]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Đã mã hóa Label thành công!")
print("Các giá trị Label sau khi mã hóa:")
print(np.unique(y_encoded))

Đã mã hóa Label thành công!
Các giá trị Label sau khi mã hóa:
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21]


## 5. Tạo bảng ánh xạ Label

Sau khi mã hóa, cần lưu lại mối quan hệ giữa:

Tên cây → Mã số

Bảng ánh xạ giúp xác định lại tên cây khi mô hình trả về kết quả dự đoán dưới dạng số.

In [6]:
mapping_df = pd.DataFrame({
    "Tên cây": label_encoder.classes_, # classes_ là nơi chứa các tên chữ của nhãn
    "Mã số": label_encoder.transform(
        label_encoder.classes_
    )
})

display(mapping_df)

,Tên cây,Mã số
0,apple,0
1,banana,1
2,blackgram,2
3,chickpea,3
4,coconut,4
5,coffee,5
6,cotton,6
7,grapes,7
8,jute,8
9,kidneybeans,9


## 6. Chia dữ liệu thành tập Train và Test

Dữ liệu được chia theo tỷ lệ:

- 80% cho tập Train.
- 20% cho tập Test.

Tập Train được sử dụng để huấn luyện mô hình.

Tập Test được sử dụng để đánh giá khả năng dự đoán trên dữ liệu mà mô hình chưa nhìn thấy.

Sử dụng `stratify=y_encoded` để duy trì tỷ lệ các loại cây giữa hai tập dữ liệu.

Dataset có 22 loại cây và mỗi loại có 100 mẫu, do đó việc giữ cân bằng giữa Train và Test là cần thiết.

X_train: Các đặc trưng đầu vào (đất, thời tiết) dùng để huấn luyện.

X_test: Các đặc trưng đầu vào dùng để kiểm thử.

y_train: Nhãn cây trồng (đã mã hóa ra số) tương ứng với tập X_train.

y_test: Nhãn cây trồng tương ứng với tập X_test.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, # tập dl đầu vào X
    y_encoded, # tập dl nhãn đã đc mã hóa thành số
    test_size=0.2, # lấy 20% dl ra để kiểm thử
    random_state=42, # làm kết quả nhất quán với nhau
    stratify=y_encoded # dl được test phải lấy đều từ các nhãn
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (1760, 7)
X_test : (440, 7)
y_train: (1760,)
y_test : (440,)


## 7. Xử lý Outlier bằng IQR Capping

Theo kết quả phân tích, các Outlier có thể là những giá trị thực tế đại diện cho điều kiện sinh trưởng của cây.

Vì vậy, thay vì xóa các dòng chứa Outlier, sử dụng phương pháp IQR Capping.

Các giá trị nhỏ hơn giới hạn dưới được đưa về giới hạn dưới.

Các giá trị lớn hơn giới hạn trên được đưa về giới hạn trên.

Các giá trị nằm trong khoảng cho phép được giữ nguyên.

Phương pháp này giúp giảm ảnh hưởng của các giá trị quá cực đoan nhưng vẫn giữ lại toàn bộ mẫu dữ liệu.

In [8]:
X_train_clean = X_train.copy()
X_test_clean = X_test.copy()

for feature in features:
    Q1 = X_train_clean[feature].quantile(0.25)
    Q3 = X_train_clean[feature].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    X_train_clean[feature] = X_train_clean[feature].clip(
        lower=lower_bound,
        upper=upper_bound
    )

    X_test_clean[feature] = X_test_clean[feature].clip(
        lower=lower_bound,
        upper=upper_bound
    )

print("Đã xử lý Outlier bằng IQR Capping!")

Đã xử lý Outlier bằng IQR Capping!


### Lưu ý

Các giới hạn IQR được tính từ tập Train.

Không sử dụng thông tin của tập Test để xác định giới hạn xử lý.

Điều này giúp tránh Data Leakage, tức là việc thông tin từ tập Test ảnh hưởng vào quá trình chuẩn bị dữ liệu huấn luyện.

## 8. Chuẩn hóa dữ liệu bằng StandardScaler

Các đặc trưng có đơn vị và phạm vi giá trị khác nhau.

Ví dụ:

- pH có khoảng giá trị nhỏ.
- rainfall có giá trị lớn hơn nhiều.
- N, P, K có thang đo riêng.

Nếu giữ nguyên thang đo, một số thuật toán Machine Learning có thể bị ảnh hưởng bởi sự khác biệt về độ lớn giữa các đặc trưng.

StandardScaler đưa các đặc trưng về cùng một thang đo dựa trên:

z = (x - μ) / σ

Trong đó:

- x là giá trị ban đầu.
- μ là giá trị trung bình của tập Train.
- σ là độ lệch chuẩn của tập Train.

In [9]:
scaler = StandardScaler() # StandardScaler kéo giá trị tb của cột về 0, độ lệch chuẩn về 1

X_train_scaled = scaler.fit_transform(X_train_clean) # tính toán giá trị tb và độ lệch chuẩn => trả về con số đã được chuẩn hóa

X_test_scaled = scaler.transform(X_test_clean) # chỉ lấy các giá trị vừa tính được để áp dụng, không cần tính toán lại bằng fit

print("Đã chuẩn hóa dữ liệu bằng StandardScaler!")

Đã chuẩn hóa dữ liệu bằng StandardScaler!


`fit_transform()` chỉ được thực hiện trên X_train.

`X_test` chỉ sử dụng `transform()`.

Không được `fit` lại StandardScaler trên X_test vì sẽ gây Data Leakage.

In [10]:
X_train = pd.DataFrame(
    X_train_scaled,
    columns=features
)

X_test = pd.DataFrame(
    X_test_scaled,
    columns=features
)

y_train = pd.Series(  # series là dl dạng 1 cột 
    y_train,
    name="label"
)

y_test = pd.Series(
    y_test,
    name="label"
)

print("Đã chuyển dữ liệu về DataFrame!")

display(X_train.head())

Đã chuyển dữ liệu về DataFrame!


,N,P,K,temperature,humidity,ph,rainfall
0,-1.371628,-1.106978,-1.038784,0.887413,0.925043,0.999688,0.246622
1,-1.127411,2.209517,2.309989,0.848496,0.426651,-1.210717,-0.663853
2,-1.073140,0.583159,-0.612188,-0.938075,-2.189453,-1.161683,0.780696
3,-0.340487,-0.469190,-0.868146,0.155436,-0.460284,-0.233750,-1.298902
4,-0.883193,-1.298314,-1.294741,-2.420728,0.916480,-0.352628,0.344354


In [11]:
print("===== KIỂM TRA DỮ LIỆU CUỐI CÙNG =====")

print("Kích thước X_train:", X_train.shape)
print("Kích thước X_test :", X_test.shape)
print("Kích thước y_train:", y_train.shape)
print("Kích thước y_test :", y_test.shape)

print("\nSố lượng giá trị thiếu:")
print(X_train.isnull().sum().sum())

print("\nSố lượng nhãn trong y_train:")
print(y_train.value_counts().sort_index())

===== KIỂM TRA DỮ LIỆU CUỐI CÙNG =====
Kích thước X_train: (1760, 7)
Kích thước X_test : (440, 7)
Kích thước y_train: (1760,)
Kích thước y_test : (440,)

Số lượng giá trị thiếu:
0

Số lượng nhãn trong y_train:
label
0     80
1     80
2     80
3     80
4     80
5     80
6     80
7     80
8     80
9     80
10    80
11    80
12    80
13    80
14    80
15    80
16    80
17    80
18    80
19    80
20    80
21    80
Name: count, dtype: int64


In [12]:
from pathlib import Path

# Tạo thư mục lưu dữ liệu đã xử lý nếu chưa tồn tại
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# Xuất 4 tập dữ liệu cuối cùng
X_train.to_csv(processed_dir / "X_train.csv", index=False)
X_test.to_csv(processed_dir / "X_test.csv", index=False)
y_train.to_csv(processed_dir / "y_train.csv", index=False)
y_test.to_csv(processed_dir / "y_test.csv", index=False)

# Lưu bảng ánh xạ nhãn để có thể chuyển mã số về tên cây
mapping_df.to_csv(processed_dir / "label_mapping.csv", index=False)

print("Đã xuất dữ liệu thành công vào:", processed_dir.resolve())
print("- X_train.csv:", X_train.shape)
print("- X_test.csv :", X_test.shape)
print("- y_train.csv:", y_train.shape)
print("- y_test.csv :", y_test.shape)
print("- label_mapping.csv:", mapping_df.shape)

Đã xuất dữ liệu thành công vào: C:\DoAnHocMay\Crop_recommendation\data\processed
- X_train.csv: (1760, 7)
- X_test.csv : (440, 7)
- y_train.csv: (1760,)
- y_test.csv : (440,)
- label_mapping.csv: (22, 2)
